# Advanced Problems with Solutions: Higher-Order Functions, `map`, `filter`, and Comprehensions

These problems focus on lazy evaluation, multiple iterables, iterator consumption, functional composition, data pipelines, and replacing `map` / `filter` with comprehensions where appropriate.

## Problem 1: Iterator Consumption

Create a `map` object that squares the numbers from `1` to `5`. Convert it to a list twice. Explain why the second result is empty.

In [1]:
numbers = [1, 2, 3, 4, 5]

squares = map(lambda x: x ** 2, numbers)

first_pass = list(squares)
second_pass = list(squares)

print(first_pass)
print(second_pass)

# Solution explanation:
# map returns an iterator. Once an iterator is consumed, it is exhausted.
# The first list(...) consumes all values. The second list(...) has nothing left to consume.

[1, 4, 9, 16, 25]
[]


## Problem 2: `map` with Unequal-Length Iterables

Use `map` to combine three iterables element-by-element. Show that `map` stops when the shortest iterable is exhausted.

In [2]:
names = ["Ana", "Ben", "Cara", "Dan"]
scores = [91, 84, 77]
grades = ["A", "B"]

result = list(map(lambda name, score, grade: {
    "name": name,
    "score": score,
    "grade": grade
}, names, scores, grades))

print(result)

# Expected output:
# [{'name': 'Ana', 'score': 91, 'grade': 'A'}, {'name': 'Ben', 'score': 84, 'grade': 'B'}]
# map stops after 2 items because grades has only 2 elements.

[{'name': 'Ana', 'score': 91, 'grade': 'A'}, {'name': 'Ben', 'score': 84, 'grade': 'B'}]


## Problem 3: Filtering Truthy Values

Use `filter(None, data)` to remove falsy values. Then explain which values were removed and why.

In [3]:
data = [0, 1, "", "Python", [], [1, 2], None, False, True, {}, {"x": 10}]

truthy_values = list(filter(None, data))

print(truthy_values)

# Falsy values removed:
# 0, "", [], None, False, {}
# These values evaluate to False in a Boolean context.

[1, 'Python', [1, 2], True, {'x': 10}]


## Problem 4: Data Cleaning Pipeline

Given a list of raw strings, remove empty or whitespace-only strings, strip remaining strings, and convert them to title case.

In [4]:
raw_names = [" alice ", "", "BOB", "   ", "cHaRlie", " dana"]

cleaned_names = list(
    map(
        lambda name: name.strip().title(),
        filter(lambda name: name.strip(), raw_names)
    )
)

print(cleaned_names)

# Alternative using a comprehension:
cleaned_names_comprehension = [
    name.strip().title()
    for name in raw_names
    if name.strip()
]

print(cleaned_names_comprehension)

['Alice', 'Bob', 'Charlie', 'Dana']
['Alice', 'Bob', 'Charlie', 'Dana']


## Problem 5: Avoid Recomputing Expensive Expressions

Rewrite the comprehension so that `x ** 2` is not computed twice.

In [5]:
# Less ideal:
result_1 = [x ** 2 for x in range(20) if x ** 2 < 100]

# Better: use map first, then filter the squared values
result_2 = list(filter(lambda square: square < 100, map(lambda x: x ** 2, range(20))))

# Better comprehension using the walrus operator, Python 3.8+
result_3 = [square for x in range(20) if (square := x ** 2) < 100]

print(result_1)
print(result_2)
print(result_3)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


## Problem 6: Function Composition

Write a higher-order function called `compose` that combines two functions. Then use it with `map`.

In [6]:
def compose(f, g):
    return lambda x: f(g(x))

def square(x):
    return x ** 2

def increment(x):
    return x + 1

increment_then_square = compose(square, increment)

result = list(map(increment_then_square, [1, 2, 3, 4]))

print(result)

# Explanation:
# compose(square, increment)(x) means square(increment(x)).
# For x = 1, result is square(2) = 4.

[4, 9, 16, 25]


## Problem 7: Filtering Dictionaries

Given a list of users, keep only active users with a score of at least 80. Then return only their usernames.

In [7]:
users = [
    {"username": "alice", "active": True, "score": 92},
    {"username": "bob", "active": False, "score": 99},
    {"username": "cara", "active": True, "score": 75},
    {"username": "dan", "active": True, "score": 88},
]

eligible_usernames = list(
    map(
        lambda user: user["username"],
        filter(lambda user: user["active"] and user["score"] >= 80, users)
    )
)

print(eligible_usernames)

# Comprehension version:
eligible_usernames_2 = [
    user["username"]
    for user in users
    if user["active"] and user["score"] >= 80
]

print(eligible_usernames_2)

['alice', 'dan']
['alice', 'dan']


## Problem 8: Custom `map`

Implement your own lazy version of `map` using `yield`. It should support multiple iterables.

In [8]:
def my_map(func, *iterables):
    iterators = [iter(iterable) for iterable in iterables]

    while True:
        try:
            values = [next(iterator) for iterator in iterators]
        except StopIteration:
            return

        yield func(*values)


result = list(my_map(lambda x, y: x + y, [1, 2, 3], [10, 20, 30]))
print(result)

short_result = list(my_map(lambda x, y: x + y, [1, 2, 3], [10]))
print(short_result)

# Like built-in map, this stops when the shortest iterable is exhausted.

[11, 22, 33]
[11]


## Problem 9: Custom `filter`

Implement your own lazy version of `filter`. It should also support `None` as the filtering function.

In [9]:
def my_filter(function, iterable):
    for item in iterable:
        if function is None:
            if item:
                yield item
        else:
            if function(item):
                yield item


print(list(my_filter(lambda x: x % 2 == 0, range(10))))
print(list(my_filter(None, [0, 1, "", "hello", [], [1]])))

[0, 2, 4, 6, 8]
[1, 'hello', [1]]


## Problem 10: Lazy Evaluation with Infinite Data

Use `itertools.count`, `map`, and `filter` to find the first 10 squares greater than 100 that are divisible by 3.

In [10]:
from itertools import count, islice

numbers = count(1)
squares = map(lambda x: x ** 2, numbers)
valid_squares = filter(lambda x: x > 100 and x % 3 == 0, squares)

first_10 = list(islice(valid_squares, 10))

print(first_10)

# This works because map and filter are lazy.
# They do not try to generate an infinite list.

[144, 225, 324, 441, 576, 729, 900, 1089, 1296, 1521]


## Problem 11: Stable Transformation Pipeline

Normalize transaction amounts, keep only positive transactions, and format them as currency strings.

In [11]:
transactions = [" 10.50", "-3.20", "0", " 99.99 ", "invalid", "42"]

def safe_float(value):
    try:
        return float(value)
    except ValueError:
        return None

amounts = map(safe_float, transactions)
positive_amounts = filter(lambda amount: amount is not None and amount > 0, amounts)
formatted = list(map(lambda amount: f"${amount:.2f}", positive_amounts))

print(formatted)

# Expected:
# ['$10.50', '$99.99', '$42.00']

['$10.50', '$99.99', '$42.00']


## Problem 12: Best-Practice Refactoring

The following code works, but it is hard to read. Refactor it using named functions and a comprehension.

In [12]:
records = [
    {"name": "alice", "score": 91, "attempts": 2},
    {"name": "bob", "score": 58, "attempts": 4},
    {"name": "cara", "score": 77, "attempts": 1},
    {"name": "dan", "score": 88, "attempts": 3},
]

# Original style:
original = list(map(lambda r: r["name"].title(), filter(lambda r: r["score"] >= 75 and r["attempts"] <= 3, records)))

print(original)

# Refactored style:
def passed_with_reasonable_attempts(record):
    return record["score"] >= 75 and record["attempts"] <= 3

def display_name(record):
    return record["name"].title()

refactored = [
    display_name(record)
    for record in records
    if passed_with_reasonable_attempts(record)
]

print(refactored)

# Best practice:
# Prefer clarity over cleverness.
# map/filter are useful, but comprehensions are often more readable for simple transformations.

['Alice', 'Cara', 'Dan']
['Alice', 'Cara', 'Dan']
